In [1]:
import os
from typing import Union

import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

In [2]:
DATA_DICT = {
    "train": "data/msc-train",
    "test":  "data/msc-test",
    "val":   "data/msc-val"
}

CLASSES = ["stop", "up"]

In [3]:
TRANSFORMATION_CFG = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.04,
    'frame_step_in_s': 0.02,
    'n_mels': 40,
    'f_min': 0,
    'f_max': 8000,
    'n_mfcc': 40,
}

TRAIN_CFG = {
    'seed': 0,
    'train_steps': 2000,
    'train_batch_size': 32
}

PRUNING_CFG = {
    # Pruning hyperparameters:
    'start_pruning': 499,     # start pruning at this training iteration
    'end_pruning': 1499,      # stop pruning after this training iteration
    'prune_amount': 0.1,      # percentage of connections to prune
    'prune_every_steps': 100, # apply pruning every N steps
}

In [4]:
class MSCDataset(torch.utils.data.Dataset):
    def __init__(self, dataPath:str, classes:list[str], transform:torch.nn.Module=None)->None:
        """Initialize the dataset by loading data from the specified path and preparing label mappings.

        Args:
            dataPath (str): Path to the dataset file.
            classes (list[str]): List of class labels.
        """
        super(MSCDataset, self).__init__()
        self.__classes = classes
        self.__transform = transform

        # creating the dictionary associating name label to integer index
        self.__convertedLabels:dict[str, int] = {label.strip().lower():i for i,label in enumerate(classes)}
        # listing all the files in the dataset path along with the converted label, accounting only for .wav files that starts with a valid label
        # sorting by folder name to ensure consistent ordering of samples
        self.__filesPath:list[list[str, int]] = sorted([[os.path.join(dataPath, file), self.__convertedLabels[file.split("_")[0].strip().lower()]]
                                                for file in os.listdir(dataPath) if file.endswith(".wav") and file.split("_")[0].strip().lower() in self.__convertedLabels],
                                                    key=lambda x: os.path.split(x[0])[0])

    @property
    def classes(self)->list[str]:
        """Get the list of class labels.

        Returns:
            classes (list[str]): List of class labels.
        """
        return list(self.__classes)


    def __len__(self)->int:
        """Return the total number of samples in the dataset.

        Returns:
            Number of samples (int): Total number of samples.
        """
        return len(self.__filesPath)


    def __getitem__(self, index:int)->dict[str, int|torch.Tensor]:
        """Retrieve a sample from the dataset at the specified index.

        Args:
            index (int): Index of the sample to retrieve.

        Returns:
            data (dict): A dictionary containing:
                - 'x' (torch.Tensor): The audio waveform tensor.
                - 'sampling_rate' (int): The sampling rate of the audio.
                - 'label' (int): The integer label corresponding to the audio class.
        """
        audio, sampling_rate = torchaudio.load(self.__filesPath[index][0])

        if audio.shape[1] < sampling_rate:
            audio = torch.nn.functional.pad(audio, (0, sampling_rate - audio.shape[1]))
        elif audio.shape[1] > sampling_rate:
            audio = audio[:, :sampling_rate]

        if self.__transform:
            audio = self.__transform(audio)

        return {
            'x': audio,
            'sampling_rate': sampling_rate,
            'label': self.__filesPath[index][1]
        }

    def label_to_int(self, label:str)->int:
        """Convert a string label to its corresponding integer label.

        Args:
            label (str): The string label to convert.
        """
        return self.__convertedLabels[label.strip().lower()]


    def getConvertedLabels(self)-> dict [int,str]:
        """Get the mapping of integer labels to string labels.

        Returns:
            convertedLabels (dict[int, str]): A dictionary mapping integer labels to string labels.
        """
        return dict(self.__convertedLabels)

    def getInvertedConvertedLabels(self)->dict[str, int]:
        """Get the mapping of string labels to integer labels.

        Returns:
            invertedConvertedLabels (dict[str, int]): A dictionary mapping string labels to integer labels.
        """
        return {v:k for k,v in self.__convertedLabels.items()}

In [5]:
transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=TRANSFORMATION_CFG['n_mfcc'],
    log_mels=True,
    melkwargs=dict(
        # Spectrogram parameters
        n_fft=int(TRANSFORMATION_CFG['frame_length_in_s'] * TRANSFORMATION_CFG['sampling_rate']),
        win_length=int(TRANSFORMATION_CFG['frame_length_in_s'] * TRANSFORMATION_CFG['sampling_rate']),
        hop_length=int(TRANSFORMATION_CFG['frame_step_in_s'] * TRANSFORMATION_CFG['sampling_rate']),
        center=False,
        # Mel Spectrogram paramaters
        f_min=TRANSFORMATION_CFG['f_min'],
        f_max=TRANSFORMATION_CFG['f_max'],
        n_mels=TRANSFORMATION_CFG['n_mels'],
    )
)

In [6]:
dataset_train = MSCDataset(dataPath=DATA_DICT["train"], classes=CLASSES, transform=transform)
dataset_test  = MSCDataset(dataPath=DATA_DICT["test"],  classes=CLASSES, transform=transform)
dataset_val   = MSCDataset(dataPath=DATA_DICT["val"],   classes=CLASSES, transform=transform)

In [ ]:
train_loader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=TRAIN_CFG['train_batch_size'],
    shuffle=True,
    num_workers=1
)

val_loader = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=TRAIN_CFG['train_batch_size'],
    shuffle=False,
    num_workers=1,
)

test_loader = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=TRAIN_CFG['train_batch_size'],
    shuffle=False,
    num_workers=1,
)

In [ ]:
class KWS_Network(nn.Module):
    """KWS Network architecture"""
    def __init__(self, n_classes: int):
        super().__init__()

        # Layer 1
        self.conv_1 = nn.Conv2d(
            in_channels=1,
            out_channels=128,
            kernel_size=(3, 3),
            stride=2,
            padding=0,
            bias=False
        )
        self.bn_conv_1 = nn.BatchNorm2d(num_features=128)
        
        # Layer 2
        self.dconv_1 = nn.Conv2d(
            in_channels=128,
            out_channels=128,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=128,
            bias=False
        )
        self.bn_dconv_1 = nn.BatchNorm2d(num_features=128)

        
        # Layer 3
        self.sconv_1 = nn.Conv2d(
            in_channels=128,
            out_channels=128,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )
        self.bn_sconv_1 = nn.BatchNorm2d(num_features=128)
        
        # Layer 4
        self.dconv_2 = nn.Conv2d(
            in_channels=128,
            out_channels=128,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=128,
            bias=False
        )
        self.bn_dconv_2 = nn.BatchNorm2d(num_features=128)
        
        # Layer 5
        self.sconv_2 = nn.Conv2d(
            in_channels=128,
            out_channels=128,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )
        self.bn_sconv_2 = nn.BatchNorm2d(num_features=128)

        # Head
        self.gap = nn.AdaptiveAvgPool2d(output_size=(1,1))
        self.flatten = nn.Flatten()
        self.fc_out = nn.Linear(in_features=128, out_features=n_classes)

        # Activation(s)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Layer 1
        l1_out = self.relu(self.bn_conv_1(self.conv_1(x)))
        # Layer 2 - should use l1_out, not x!
        l2_out = self.relu(self.bn_dconv_1(self.dconv_1(l1_out)))
        # Layer 3 - should use l2_out, not x!
        l3_out = self.relu(self.bn_sconv_1(self.sconv_1(l2_out)))
        # Layer 4 - should use l3_out, not x!
        l4_out = self.relu(self.bn_dconv_2(self.dconv_2(l3_out)))
        # Layer 5 - should use l4_out, not x!
        l5_out = self.relu(self.bn_sconv_2(self.sconv_2(l4_out)))
        # Head
        out = self.fc_out(self.flatten(self.gap(l5_out)))
        return out

In [9]:
def traininig_step(model, loss_fn, optimizer, train_loader, device):
    model.train()
    train_loss = 0
    for data in train_loader:
        X, y = data["x"], data["label"]
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(X)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    return train_loss / len(train_loader)

def evaluation_step(model, loss_fn, val_loader, device):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for data in val_loader:
            X, y = data["x"], data["label"]
            X, y = X.to(device), y.to(device)
            output = model(X)
            loss = loss_fn(output, y)
            val_loss += loss.item()

    return val_loss / len(val_loader)

In [10]:
from torch.optim import optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device: %s\n"%device)
model = KWS_Network(len(CLASSES)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.75, patience=5, min_lr = 1e-7

)

device: cpu



In [ ]:
# Calculate epochs needed to reach 2000 training steps
steps_per_epoch = len(train_loader)
total_epochs = TRAIN_CFG['train_steps'] // steps_per_epoch

print(f"Dataset size: {len(dataset_train)}\n")
print(f"Steps per epoch: {steps_per_epoch}\n")
print(f"Training for {total_epochs} epochs to reach {TRAIN_CFG['train_steps']} steps\n")

# Training loop
best_val_loss = float('inf')
best_model_path = 'best_model.pth'

for epoch in range(total_epochs):
    avg_train_loss = traininig_step(model, loss_fn, optimizer, train_loader, device)
    avg_eval_loss = evaluation_step(model, loss_fn, val_loader, device)
    scheduler.step(avg_eval_loss)
    
    # Save best model
    if avg_eval_loss < best_val_loss:
        best_val_loss = avg_eval_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_eval_loss,
        }, best_model_path)
        print(f"✓ Epoch {epoch + 1}/{total_epochs} | NEW BEST | Train: {avg_train_loss:.4f} | Val: {avg_eval_loss:.4f}")
    else:
        print(f"  Epoch {epoch + 1}/{total_epochs} | Train: {avg_train_loss:.4f} | Val: {avg_eval_loss:.4f}")

Dataset size: 1600

Steps per epoch: 50

Training for 40 epochs to reach 2000 steps

